(ex-bedload-arbogne)=
# Geschiebe an der Arbogne: Korn, Rauheit und Shields-Diagramm

Zum interaktiven Lesen und Ausführen der Code-Blöcke [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hydro-informatics/jupyter-python-course/main?labpath=bedload-exercise.ipynb), oder das Notebook herunterladen [![GitHub](https://img.shields.io/badge/GitHub-download-181717?logo=github)](https://github.com/hydro-informatics/jupyter-python-course/blob/main/bedload-exercise.ipynb) und {ref}`Python <install-python>` sowie {ref}`JupyterLab <jupyter>` lokal installieren, um *bedload-exercise.ipynb* auf dem eigenen Rechner auszuführen. Das Notebook auf Binder und GitHub ist die französische Originalfassung dieser Seite.

```{important}
Die Übung besteht aus drei Dateien, die **diese Ordnerstruktur** beibehalten müssen. Nicht die Zip-Datei des Repositorys [jupyter-python-course](https://github.com/hydro-informatics/jupyter-python-course) herunterladen (sie enthält alle Notebooks und Datensätze des Kurses), sondern die drei Dateien **einzeln** (auf GitHub: *Download raw file*).

| Datei | Inhalt |
|---|---|
| [*bedload-exercise.ipynb*](https://github.com/hydro-informatics/jupyter-python-course/blob/main/bedload-exercise.ipynb) | dieses Notebook, die einzige zu bearbeitende Datei |
| [*fun/charriage.py*](https://github.com/hydro-informatics/jupyter-python-course/blob/main/fun/charriage.py) | der technische Code: Formel von Meyer-Peter & Müller (MPM), Kontrolle und Abbildungen |
| [*data/hecras-arbogne.csv*](https://github.com/hydro-informatics/jupyter-python-course/blob/main/data/hecras-arbogne.csv) | die Hydraulik von 16 HEC-RAS-1D-Querprofilen der Arbogne bei Dompierre (FR) für Q = 25 m$^3$/s |
```

```{admonition} Aufgabe
Ihre Expertise ist gefragt, um das Risiko von Auflandungen im revitalisierten Abschnitt der Arbogne zu analysieren. Ein Kollege hat Ihnen die (1D-)Hydraulik von 16 Querprofilen auf 890 m geliefert. Wo bleibt der Kies liegen, und was passiert, wenn Vegetation die Strömung bremst?
```

## Aufbau der Übung

| | Teil | wer arbeitet |
|---|---|---|
| **I** | Vorbereitung: Skript und Daten laden | **gegeben**: Zellen ausführen |
| **II** | Daten: Kornverteilung, Rauheiten $k_{st}$, Schwelle $\tau_{*,cr}$ | **Sie**: zwei Listen ergänzen |
| **III** | Gleichungen: Rauheitsverhältnis $k'$ und kritische Rauheit $k_{st,cr}$ | **Sie**: zwei Codezeilen ergänzen |
| **IV** | Ergebnisse: MPM-Berechnung des {term}`Geschiebetransports <Bedload>`, Shields-Diagramm, aktive Querprofile | **gegeben**: ausführen und interpretieren |

Programmierkenntnisse sind nicht erforderlich. Eine Zelle wird mit **Umschalt + Eingabe** (*Shift + Enter*) ausgeführt. Die zu ergänzenden Stellen sind mit dem Kommentar `# >>> À COMPLÉTER` (*zu ergänzen*) markiert: Ersetzen Sie jeden Text `">>> COMPLETER"` (inklusive Anführungszeichen) durch Ihren Wert oder Ihre Gleichung. Im Code werden Dezimalzahlen mit einem **Punkt** geschrieben (0.047 und nicht 0,047).

```{note}
Der Code ist französisch, wie das Notebook, das Binder startet: Variablen- und Funktionsnamen, Kommentare und die Meldungen von *charriage.py*. Die Namen, mit denen Sie arbeiten, sind `melanges` (Korngemische), `k_st` (Strickler-Beiwerte), `tau_cr` (kritischer Shields-Parameter), `rapport_rugosite` (Rauheitsverhältnis) und `k_st_critique` (kritische Rauheit).
```

---

# Teil I: Vorbereitung

Die folgende Zelle lädt das Skript *fun/charriage.py* unter dem Kurznamen `ch` und zeigt die Daten aus *data/hecras-arbogne.csv* an. Diese Zelle enthält keinen Lernstoff.

In [ ]:
from fun import charriage as ch   # fun/charriage.py, lit data/hecras-arbogne.csv
ch.lire_hecras()

Die Tabelle zeigt für jedes Querprofil (in Fließrichtung) die Station, den Abfluss, die Sohl- und Wasserspiegelkoten, das Energieliniengefälle $J_e$, den hydraulischen Radius $R_h$ und die Abflusstiefe $h$. Das Gefälle $J_e$ nimmt von 0,0042 oberstrom auf 0,0007 unterstrom ab: Dort droht sich der Kies abzulagern.

---

# Teil II: Daten

## Aufgabe 1: Kornverteilung und Rauheit

Zwei Kornverteilungskurven beschreiben die Arbogne (Linienzahlanalyse nach Fehr, März 2014):

- **„Sable (fin)“** (Sand, fein), um den **Eintrag** von oberstrom abzuschätzen;
- **„Gravier (complet)“** (Kies, vollständig), um die anstehende **Sohle** zu charakterisieren.

Drei charakteristische Korndurchmesser genügen für die Übung:

| Kornverteilungskurve | $d_m$ [mm] | $d_{84}$ [mm] |
|---|---|---|
| „Sable (fin)“ | 0,23 | nicht benötigt |
| „Gravier (complet)“ | 8,88 | 13,56 |

- $d_m$ des Sandes steht für den durchlaufenden Feinanteil;
- $d_m$ des Kieses steht für den transportierten Kies;
- $d_{84}$ des Kieses steht für die anstehende Sohle (Deckschicht) und bestimmt die Kornrauheit $k_r$.

Tragen Sie diese Werte **in Millimetern** in die Liste `melanges` ein. Achtung: In Python wird das Dezimalkomma zum **Punkt** (0,23 wird `0.23` geschrieben). Die Reihenfolge zählt: zuerst der Sand, dann der Kies (das letzte Gemisch der Liste steht für die Sohle).

Tragen Sie anschließend in die Liste `k_st` die beiden zu vergleichenden Strickler-Beiwerte $k_{st}$ [m$^{1/3}$/s] ein, vom glattesten zum rauesten:

1. **ohne Vegetation**: Die Strömung bleibt im Kiesbett (Referenzzustand);
2. **mit Vegetation**: Das Hochwasser ufert in das dicht bewachsene Vorland aus.

Die Liste darf weitere Werte enthalten: Jeder zusätzliche $k_{st}$ wird zu einem weiteren Rauheitsfall.

In [ ]:
# >>> À COMPLÉTER : diamètres caractéristiques en mm
#             nom                   d_m               d_84
melanges = [["Sable (fin)",         ">>> COMPLETER"],
            ["Gravier (complet)",   ">>> COMPLETER",  ">>> COMPLETER"]]

# >>> À COMPLÉTER : coefficients de Strickler k_st [m^(1/3)/s], sans végétation en premier
k_st = [">>> COMPLETER", ">>> COMPLETER"]

## Schwelle des Bewegungsbeginns $\tau_{*,cr}$

MPM wurde mit einem kritischen {term}`Shields-Parameter <Shields parameter>` von $\tau_{*,cr}$ = 0,047 kalibriert; dieser Wert gilt für die ganze Übung. Die Liste `tau_cr` akzeptiert trotzdem mehrere Werte, zum Beispiel um die Empfindlichkeit des Ergebnisses gegenüber der Schwelle zu testen (siehe Hausaufgabe). Entfernen Sie dazu das `#` vor der zweiten Zeile: Jede Abbildung zeigt dann alle Werte der Liste.

```{note}
$\tau_{*,cr}$ lässt sich nicht messen. Man wählt den Wert innerhalb einer Bandbreite (etwa 0,03 bis 0,07 für Kies) und gibt ihn zusammen mit dem Ergebnis an.
```

In [ ]:
tau_cr = [0.047]
# tau_cr = [0.030, 0.040, 0.047, 0.056, 0.070]   # variante, pas utilisée dans l'exercice

---

# Teil III: Gleichungen

## Aufgabe 2: Rauheitsverhältnis $k'$

MPM vergleicht nicht die gesamte {term}`dimensionslose Sohlschubspannung <Dimensionless bed shear stress>` $\tau_*$ mit der Schwelle, sondern nur den Anteil, der auf die Körner wirkt. Ein Teil der Energie wird durch Sohlformen, Vegetation und Ufer dissipiert. Das Rauheitsverhältnis $k'$ korrigiert $\tau_*$ entsprechend:

$$\Phi = 8\,\left(k'\,\tau_* - \tau_{*,cr}\right)^{3/2} \qquad \text{mit} \qquad \tau_* = \frac{R_h\,J_e}{(s-1)\,d}$$

$$\boxed{\;k' = \left(\frac{k_{st}}{k_r}\right)^{3/2} \le 1\;} \qquad \text{wobei} \qquad k_r = \frac{26}{d_{84}^{1/6}}$$

- $k_{st}$: **gesamter** Strickler-Beiwert des Abschnitts (die Liste `k_st` aus Aufgabe 1);
- $k_r$: Strickler-Beiwert **des Korns** einer ebenen Sohle, berechnet von *charriage.py* mit $d_{84}$ des Kieses in Metern.

Ergänzen Sie die markierte Zeile. In Python wird eine Potenz mit `**` geschrieben: Zum Beispiel wird $a^{3/2}$ als `a ** (3 / 2)` geschrieben.

In [ ]:
def rapport_rugosite(k_st, k_r):
    # >>> À COMPLÉTER : k' = (k_st / k_r)^(3/2)
    return ">>> COMPLETER"

## Aufgabe 3: kritische Rauheit $k_{st,cr}$

Je stärker die Vegetation die Strömung bremst, desto kleiner wird $k_{st}$ und mit ihm die wirksame Schubspannung $k'\,\tau_*$. Ein Querprofil hört auf zu transportieren, wenn $k'\,\tau_* = \tau_{*,cr}$. Ersetzt man $k'$ durch seine Definition und löst nach $k_{st}$ auf, erhält man die kritische Rauheit jedes Querprofils:

$$\boxed{\;k_{st,cr} = k_r \left(\frac{\tau_{*,cr}}{\tau_*}\right)^{2/3}\;}$$

Ein Querprofil transportiert, solange $k_{st} > k_{st,cr}$. Mit dieser Gleichung lässt sich die Anzahl aktiver Querprofile bei zunehmender Rauheit darstellen (letzte Abbildung). Ergänzen Sie die markierte Zeile.

In [ ]:
def k_st_critique(tau, k_r, tau_cr):
    # >>> À COMPLÉTER : k_st,cr = k_r * (tau_cr / tau)^(2/3)
    return ">>> COMPLETER"

## Kontrolle

Führen Sie diese Zelle nach jeder Aufgabe aus. `[OK]` bedeutet, dass der Wert mit der Vorlesung übereinstimmt, `[XX]`, dass er falsch ist, und `[  ]`, dass er noch nicht ergänzt ist. Korrigieren Sie den Wert und führen Sie dann die Zelle der Aufgabe **und** diese Zelle erneut aus.

In [ ]:
ch.verifier(melanges, k_st, rapport_rugosite, k_st_critique)

---

# Teil IV: Ergebnisse

## MPM-Berechnung für die 16 Querprofile

*charriage.py* kombiniert Ihre Daten und wendet MPM auf jedes Querprofil an: Jeder Rauheitsfall der Liste `k_st` wird für beide Kornklassen berechnet.

| Korn | was es darstellt | Rauheitsfälle |
|---|---|---|
| Kies, $d_m$ | der transportierte Kies | ohne und mit Vegetation |
| Sand, $d_m$ | der Feineintrag von oberstrom | ohne und mit Vegetation |
| Kies, $d_{84}$ | die anstehende, abgepflasterte Sohle | nur ohne Vegetation |

Die Tabelle fasst für jede Kombination das Rauheitsverhältnis $k'$, die Anzahl der Querprofile, die noch transportieren ($\Phi > 0$), sowie das größte $\Phi$ und die größte Geschiebetransportrate $q_b$ [kg/(s$\cdot$m)] des Abschnitts zusammen.

In [ ]:
resultats = ch.calculer(melanges, k_st, tau_cr, rapport_rugosite)
ch.resume(resultats)

```{admonition} Frage
Warum ist $\Phi$ = 0 an einigen Querprofilen? Und warum wird der Sand überall transportiert?
```

## Shields-Diagramm

Das Diagramm vergleicht die beiden Rauheitsfälle **ohne Vegetation** (Abfluss im Gerinne) und **mit Vegetation** (Hochwasser ufert in das Vorland aus) für beide Kornklassen, **Kies und Sand**. Jeder Punkt ist ein Querprofil. Die vertikale Achse ist die wirksame Schubspannung auf die Körner $k'\,\tau_*$, also diejenige, die MPM mit $\tau_{*,cr}$ vergleicht. Unterhalb der kritischen Shields-Kurve findet kein Geschiebetransport statt.

- **Farbe**: Rauheitsfall (blau ohne Vegetation, rot mit Vegetation);
- **Symbol**: Kornklasse (Kreis: Kies; Dreieck: Sand);
- **Größe**: $\Phi$, eine Dekade pro Klasse (das größte $\Phi$ ergibt die größte Markierung).

In [ ]:
ch.diagramme_shields(resultats)

```{note}
Mit $d_m$ = 0,23 mm liegt der Sand außerhalb des Gültigkeitsbereichs von MPM (etwa 0,4 bis 30 mm). Sein Ergebnis ist nur eine Größenordnung.
```

```{admonition} Fragen
Verschiebt die Vegetation den Kies und den Sand auf die gleiche Weise? Bei welcher Kornklasse lässt die Vegetation die Punkte unter die Kurve fallen?
```

## Aktive Querprofile bei zunehmender Rauheit

Die folgende Abbildung verringert $k_{st}$ schrittweise (von links nach rechts nimmt die Vegetation zu) und zählt für den Kies die Querprofile, in denen $k_{st} > k_{st,cr}$. Die vertikalen Linien sind Ihre Werte aus der Liste `k_st`. Der **vollständige Stillstand** ist der $k_{st}$-Wert, unterhalb dessen kein Querprofil mehr transportiert.

In [ ]:
ch.profils_actifs(melanges, k_st, tau_cr, k_st_critique)

```{admonition} Fragen
1. Zwischen welchen Werten von $k_{st}$ kommt der Geschiebetransport des Kieses vollständig zum Erliegen?
2. Auflandung oder Erosion (hier: Erhalt eines Gerinnes an der Bewegungsgrenze, *threshold channel*): Welcher der beiden Parameter, das Korn oder die Rauheit, gibt den Ausschlag?
```